In [0]:
storage_account = "adlsgen2dreamprod"
client_id = dbutils.secrets.get(scope="akv-scope", key="client-id")
client_secret = dbutils.secrets.get(scope="akv-scope", key="client-secret")
tenant_id = dbutils.secrets.get(scope="akv-scope", key="tenant_id")
spark.conf.set(f"fs.azure.account.oauth.type.{storage_account}.dfs.core.windows.ney","OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               "org.apache.hadoop.bfs.oauth2.ClientCredentialTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client-id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client-secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
df_silver_path = f"abfs://silver@{storage_account}.dfs.core.windows.net"
df_silver_locations = spark.read.format("delta").load(df_silver_path)
df_silver_locations.createOrReplaceTempView("silver_locations_view")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;
use gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dim_locations (
    location_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    location_id STRING,
    city STRING,
    state STRING,
    country STRING,
    latitude DECIMAL(11,8),
    longitude DECIMAL(11,8),
    last_updated_timestamp TIMESTAMP,
    scd1_hash STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    _processed_at TIMESTAMP
) USING DELTA
LOCATION 'abfss://gold@adlsgen2dreampod.dfs.core.windows.net/dream_app/dim_locations';
    
CREATE OR REPLACE TEMPORARY VIEW v_locations_source AS
SELECT
    location_id,
    city,
    state,
    country,
    latitude,
    longitude,
    last_updated_timestamp,
    md5(concat_ws('|', 
        upper(trim(city)), 
        upper(trim(state)), 
        upper(trim(country)), 
        latitude, 
        longitude
    )) AS source_hash,
    current_timestamp() AS _processed_at
FROM
    silver_locations_view;
    
MERGE INTO dim_locations AS target
USING v_locations_source AS source
ON target.location_id = source.location_id
WHEN MATCHED AND target.scd1_hash <> source.source_hash THEN
    UPDATE SET
        target.city = source.city,
        target.state = source.state,
        target.country = source.country,
        target.latitude = source.latitude,
        target.longitude = source.longitude,
        target.scd1_hash = source.source_hash,
        target._processed_at = source._processed_at,
        target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (location_id, city, state, country, latitude, longitude, scd1_hash, _processed_at, created_at, updated_at)
    VALUES (source.location_id, source.city, source.state, source.country, source.latitude, source.longitude, source.source_hash, source._processed_at, current_timestamp(), current_timestamp());